In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler ,LabelEncoder
import pickle

In [6]:
data = pd.read_csv('Churn_Modelling.csv')
data =data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [7]:
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [8]:
from sklearn.preprocessing import OneHotEncoder
one_hot_encoder_geo = OneHotEncoder()
geo_encoder = one_hot_encoder_geo.fit_transform(data[['Geography']])
one_hot_encoder_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [14]:
geo_encoded_df = pd.DataFrame(geo_encoder.toarray(), columns=one_hot_encoder_geo.get_feature_names_out(['Geography']))
data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)


In [10]:
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('one_hot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(one_hot_encoder_geo, file)    
    

In [15]:
X = data.drop('Exited', axis=1)
Y = data['Exited']
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [16]:
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)
    

In [3]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime


In [17]:
Model = Sequential(
    [
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
    ]
)

In [18]:
Model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2,945
Trainable params: 2,945
Non-trainable params: 0
_________________________________________________________________


In [19]:
opt = tf.keras.optimizers.Adam(learning_rate=0.01)
loss = tf.keras.losses.BinaryCrossentropy()


In [21]:
Model.compile(optimizer=opt, loss=loss, metrics=['accuracy'])

In [22]:
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [ ]:
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [25]:
history = Model.fit(
    X_train_scaled, Y_train,
    validation_data=(X_test_scaled, Y_test),
    epochs=50,
    batch_size=32,
    callbacks=[early_stopping_callback, tensorboard_callback]
)

Epoch 1/50
250/250 [==============================] - 0s 1ms/step - loss: 0.3380 - accuracy: 0.8627 - val_loss: 0.3445 - val_accuracy: 0.8605
Epoch 2/50
250/250 [==============================] - 0s 1ms/step - loss: 0.3374 - accuracy: 0.8640 - val_loss: 0.3379 - val_accuracy: 0.8590
Epoch 3/50
250/250 [==============================] - 0s 1ms/step - loss: 0.3344 - accuracy: 0.8612 - val_loss: 0.3566 - val_accuracy: 0.8550
Epoch 4/50
250/250 [==============================] - 0s 1ms/step - loss: 0.3307 - accuracy: 0.8654 - val_loss: 0.3451 - val_accuracy: 0.8550
Epoch 5/50
250/250 [==============================] - 0s 1ms/step - loss: 0.3288 - accuracy: 0.8660 - val_loss: 0.3511 - val_accuracy: 0.8570
Epoch 6/50
250/250 [==============================] - 0s 1ms/step - loss: 0.3247 - accuracy: 0.8675 - val_loss: 0.3362 - val_accuracy: 0.8585
Epoch 7/50
250/250 [==============================] - 0s 1ms/step - loss: 0.3241 - accuracy: 0.8683 - val_loss: 0.3517 - val_accuracy: 0.8585
Epoch 

In [26]:
Model.save('Model.h5')

In [50]:
%reload_ext tensorboard

In [1]:
%load_ext tensorboard
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 24144), started 0:21:20 ago. (Use '!kill 24144' to kill it.)

In [2]:
print("hii")

hii
